# Regression Explained — Linear, Lasso, Ridge, Polynomial, ElasticNet

Companion to `linerREG.ipynb`. This notebook is **theory and decision-making** — formulas, what each method optimizes, when to reach for which one, and pros/cons. No model training here.

**Notation used throughout**

- $n$ = number of training samples, $p$ = number of features
- $x_i \in \mathbb{R}^p$ = feature vector for sample $i$
- $y_i \in \mathbb{R}$ = target for sample $i$
- $\hat{y}_i$ = predicted target
- $\beta = (\beta_0, \beta_1, \dots, \beta_p)$ = coefficients (weights), with $\beta_0$ the intercept
- $\bar{y} = \frac{1}{n}\sum_i y_i$ = mean of the target

## 1. The standard regression workflow

Every regression project, no matter the algorithm, follows roughly the same pipeline:

```
raw data
   │
   ▼
(1) Inspect & clean        ── nulls, duplicates, dtypes, obvious outliers
   │
   ▼
(2) EDA                    ── target distribution, feature/target correlation, multicollinearity
   │
   ▼
(3) Encode categoricals    ── one-hot / ordinal / binary 0-1
   │
   ▼
(4) Train / test split     ── usually 80/20, stratified only for classification
   │
   ▼
(5) Scale features         ── REQUIRED for Lasso / Ridge / ElasticNet,
   │                         otherwise the penalty unfairly punishes
   │                         large-magnitude features
   ▼
(6) Fit baseline           ── plain LinearRegression first — it's your floor
   │
   ▼
(7) Try regularized variants if needed
   │   ├── many correlated / noisy features      → Ridge
   │   ├── want feature selection (sparse model) → Lasso
   │   ├── correlated groups + want sparsity     → ElasticNet
   │   └── relationship looks curved             → Polynomial
   ▼
(8) Tune α (and l1_ratio)  ── via cross-validation, e.g. RidgeCV / LassoCV
   │
   ▼
(9) Evaluate on test set   ── MSE, RMSE, R² + residual plot
   │
   ▼
(10) Sanity-check          ── predicted-vs-actual, residuals random?
```

Two rules to internalize:

1. **Always start with plain Linear Regression.** If it already explains 99% of variance (as in the student-performance data), regularization or polynomial expansion just adds complexity without gain.
2. **Always scale before regularizing.** $L_1$/$L_2$ penalties shrink coefficient magnitudes. If `Income` is in dollars and `Age` is in years, the penalty effectively sees them on different planets.

## 2. Linear Regression (Ordinary Least Squares)

### Model
$$
\hat{y}_i = \beta_0 + \beta_1 x_{i1} + \beta_2 x_{i2} + \dots + \beta_p x_{ip} = x_i^\top \beta
$$

### Loss function (the thing it minimizes)
$$
\mathcal{L}_{\text{OLS}}(\beta) \;=\; \sum_{i=1}^{n} (y_i - \hat{y}_i)^2
$$

### Closed-form solution
Writing $X \in \mathbb{R}^{n \times (p+1)}$ as the design matrix (with a column of 1s for the intercept):
$$
\hat{\beta} = (X^\top X)^{-1} X^\top y
$$
This is the so-called **normal equation**. It exists in closed form — no iteration needed — *as long as $X^\top X$ is invertible*. That fails when features are perfectly collinear.

### Classical OLS assumptions
1. **Linearity** — the true relationship is linear in $\beta$.
2. **Independence** — observations are independent of each other.
3. **Homoscedasticity** — error variance is constant across all $x$.
4. **Normally distributed errors** — needed for valid p-values / CIs, *not* for the point estimate.
5. **No perfect multicollinearity** — features aren't exact linear combinations of each other.

If these are violated, the coefficients are still computable but standard errors and tests become misleading.

### Pros
- Closed-form, instant to fit.
- Coefficients are directly interpretable ("+1 hour studied → +7.4 points").
- No hyperparameters to tune.
- Strong baseline; often hard to beat on well-behaved data.

### Cons
- Breaks down with multicollinearity (coefficients become unstable, can flip sign with small data changes).
- Sensitive to outliers — the squared loss heavily weights extreme points.
- Overfits when $p$ approaches $n$ or features are noisy.
- Can't capture non-linear relationships on its own.

### Reach for it when
Features are roughly independent, the target looks linearly related to features, and $n \gg p$.

## 3. Ridge Regression (L2 regularization)

### Loss function
$$
\mathcal{L}_{\text{Ridge}}(\beta) \;=\; \underbrace{\sum_{i=1}^{n} (y_i - x_i^\top \beta)^2}_{\text{OLS loss}} \;+\; \alpha \underbrace{\sum_{j=1}^{p} \beta_j^2}_{L_2\text{ penalty}}
$$

$\alpha \geq 0$ is the regularization strength. $\alpha = 0$ → identical to OLS. $\alpha \to \infty$ → all coefficients shrink to 0.

### Closed-form solution
$$
\hat{\beta}_{\text{Ridge}} = (X^\top X + \alpha I)^{-1} X^\top y
$$
Adding $\alpha I$ to the diagonal makes the matrix invertible *even under multicollinearity* — that's the algebraic reason Ridge fixes OLS's biggest failure mode.

### Geometric intuition
Ridge constrains the coefficient vector to lie inside a sphere (a ball in coefficient space). The optimizer finds the OLS solution that also stays inside that ball. Smaller $\alpha$ = bigger ball = closer to OLS.

### Pros
- Handles multicollinearity gracefully — coefficients become stable.
- Reduces variance at the cost of a small bias (good bias–variance trade for noisy data).
- Closed-form, fast.
- Keeps all features in the model (no zero coefficients).

### Cons
- Doesn't do feature selection — every feature gets a (possibly tiny) non-zero weight.
- Need to scale features first.
- $\alpha$ must be tuned (use `RidgeCV`).

### Reach for it when
You have many features, several of them correlated, and you don't need a sparse model — you just want stable, generalizable coefficients.

## 4. Lasso Regression (L1 regularization)

### Loss function
$$
\mathcal{L}_{\text{Lasso}}(\beta) \;=\; \sum_{i=1}^{n} (y_i - x_i^\top \beta)^2 \;+\; \alpha \sum_{j=1}^{p} |\beta_j|
$$

Same OLS data-fit term, but the penalty is the *absolute value* of coefficients (the $L_1$ norm) instead of squared.

### Why this changes everything
The $L_1$ ball has corners on the axes. The optimum often lands exactly on a corner, which forces some coefficients to be **exactly zero**. That's automatic feature selection — Lasso both regularizes *and* removes features.

There's no closed form (the absolute value isn't differentiable at 0). Solvers use coordinate descent or LARS.

### Pros
- Built-in feature selection — final model uses only the predictors that matter.
- Produces sparse, interpretable models.
- Handles high-dimensional data ($p > n$) reasonably well.

### Cons
- When two features are highly correlated, Lasso arbitrarily picks one and zeros the other — unstable selection.
- Can over-shrink coefficients of important features.
- No closed form; iterative solver.
- Need to scale features first.

### Reach for it when
You have many features but suspect only a handful actually matter, and you want the model to tell you which ones.

## 5. ElasticNet Regression (L1 + L2)

### Loss function
$$
\mathcal{L}_{\text{Enet}}(\beta) = \sum_{i=1}^{n}(y_i - x_i^\top \beta)^2 \;+\; \alpha \left[\, \rho \sum_{j=1}^{p} |\beta_j| + \tfrac{1-\rho}{2} \sum_{j=1}^{p} \beta_j^2 \,\right]
$$

Two hyperparameters:
- $\alpha$ — overall regularization strength.
- $\rho \in [0, 1]$ — mixing ratio (`l1_ratio` in scikit-learn). $\rho = 1$ is pure Lasso, $\rho = 0$ is pure Ridge.

### Why mix them
Lasso's weakness is correlated features (it picks one and drops the rest). Ridge's weakness is no sparsity (it keeps everything). ElasticNet does both: the $L_2$ term encourages correlated features to be selected together, while the $L_1$ term still drives some coefficients to zero.

### Pros
- Best of both worlds — sparsity with stability across correlated features.
- Particularly strong when $p > n$ or when features cluster into correlated groups.

### Cons
- Two hyperparameters to tune ($\alpha$ and `l1_ratio`).
- More expensive to cross-validate than Ridge or Lasso alone.
- If you don't have correlated groups, plain Lasso or Ridge is usually simpler.

### Reach for it when
Your features form correlated clusters (e.g., genomic data, text features, financial indicators) and you want both selection and stability.

## 6. Polynomial Regression

### Model
Polynomial regression isn't really a different *algorithm* — it's a feature engineering trick. You expand $x$ into polynomial terms and then run ordinary linear regression on the expanded features:

For one feature, degree 2:
$$
\hat{y} = \beta_0 + \beta_1 x + \beta_2 x^2
$$

For two features ($x_1, x_2$), degree 2:
$$
\hat{y} = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \beta_3 x_1^2 + \beta_4 x_1 x_2 + \beta_5 x_2^2
$$

Note the **interaction term** $x_1 x_2$ — this is often the real value of polynomial expansion, not the squared terms.

### Feature explosion
For $p$ features at degree $d$, the number of polynomial terms is $\binom{p+d}{d}$. With $p=10$ features and degree 4, that's 1,001 terms. Easy to overfit.

### Pros
- Captures non-linear relationships using simple linear-regression machinery.
- Adds interaction effects between features automatically.
- Still interpretable at low degrees.

### Cons
- Feature count blows up with degree.
- High-degree fits oscillate wildly between/outside training points (Runge's phenomenon).
- Extrapolation is dangerous — predictions outside the training range explode.
- Often pair with Ridge to tame the variance: `PolynomialFeatures` → `Ridge`.

### Reach for it when
EDA shows clearly curved relationships and you want a small, interpretable model. Stop at degree 2 or 3 unless you have a strong reason. For genuinely complex non-linearities, prefer trees / gradient boosting.

## 7. Evaluation metrics

### Mean Squared Error (MSE)
$$
\text{MSE} = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2
$$

- Always $\geq 0$, lower is better.
- Units are the **square** of the target (hard to interpret directly).
- Heavily penalizes large errors — outlier-sensitive.

### Root Mean Squared Error (RMSE)
$$
\text{RMSE} = \sqrt{\text{MSE}} = \sqrt{\frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2}
$$

- Same units as the target. "On average, the prediction is off by RMSE units."
- Still outlier-sensitive (it's a monotonic transform of MSE).
- Most-cited metric in regression reports.

### Coefficient of Determination ($R^2$)
$$
R^2 \;=\; 1 \;-\; \frac{\sum_{i}(y_i - \hat{y}_i)^2}{\sum_{i}(y_i - \bar{y})^2} \;=\; 1 - \frac{\text{SS}_{\text{res}}}{\text{SS}_{\text{tot}}}
$$

- Fraction of variance in $y$ explained by the model.
- $R^2 = 1$ → perfect fit. $R^2 = 0$ → model is no better than predicting the mean. $R^2 < 0$ → model is *worse* than predicting the mean (yes, this is possible on a held-out test set).
- Unitless — easy to compare across datasets/scales.
- Caveat: $R^2$ never decreases when you add features to the *training* set, even useless ones. Use **adjusted $R^2$** if comparing models with different feature counts on the same data.

### Which to report
- **RMSE** for "how big is the typical error?" (in actual units).
- **$R^2$** for "how much of the signal does the model capture?" (unitless).
- Both, side by side, is the standard.

## 8. Decision guide — which model when?

```
Q1: Is the relationship roughly linear?
    │
    ├── No, clearly curved
    │       └── PolynomialFeatures + (Ridge | LinearRegression)
    │           └── still bad?  → tree-based models (RandomForest, XGBoost)
    │
    └── Yes, looks linear
            │
            ▼
Q2: How many features vs samples?  And are features correlated?
    │
    ├── n >> p, features mostly independent
    │       └── LinearRegression  (don't over-engineer)
    │
    ├── Many correlated features, not seeking sparsity
    │       └── Ridge
    │
    ├── Many features, suspect only a few matter
    │       └── Lasso
    │
    ├── Many features, correlated groups, want sparsity AND stability
    │       └── ElasticNet
    │
    └── p > n  (wider than tall)
            └── Lasso or ElasticNet (OLS won't even fit)
```

### Quick mental shortcuts

| You see this in your data...                     | First model to try |
|---------------------------------------------------|--------------------|
| Few clean features, lots of rows                  | LinearRegression   |
| High variance inflation factor (VIF > 10)         | Ridge              |
| 100+ features, want a simple final model          | Lasso              |
| Genomic / text / sensor data with feature groups  | ElasticNet         |
| Scatter plot shows a clear curve                  | Polynomial (deg 2) |
| Outliers dominate MSE                             | Try Huber / RANSAC |
| Target is bounded (0–1) or count                  | Logistic / Poisson — *not* linear regression |

## 9. Side-by-side comparison

| Aspect              | Linear (OLS) | Ridge ($L_2$) | Lasso ($L_1$) | ElasticNet | Polynomial |
|---------------------|--------------|---------------|---------------|------------|------------|
| Penalty term        | none         | $\alpha\sum\beta_j^2$ | $\alpha\sum\|\beta_j\|$ | mix of $L_1$+$L_2$ | none (feature trick) |
| Closed form?        | yes          | yes           | no (coord. descent) | no | yes (after expansion) |
| Feature selection?  | no           | no            | **yes**       | **yes**    | no |
| Handles multicollinearity? | no    | **yes**       | partially     | **yes**    | depends on base model |
| Captures non-linearity? | no       | no            | no            | no         | **yes** |
| Hyperparameters     | 0            | 1 ($\alpha$)  | 1 ($\alpha$)  | 2 ($\alpha$, $\rho$) | 1 (degree) |
| Needs feature scaling? | not strictly | **yes**     | **yes**       | **yes**    | yes (after expansion) |
| Interpretability    | high         | high          | very high (sparse) | high  | medium (degree↑ → ↓) |
| Risk of overfitting | medium       | low           | low           | low        | high if degree too big |

## 10. Applying this to the student-performance result

Recall the test-set scores from `linerREG.ipynb`:

| Model                      | RMSE   | R²     |
|----------------------------|--------|--------|
| Polynomial (deg=2)         | 2.0749 | 0.9884 |
| Linear Regression          | 2.0751 | 0.9884 |
| Ridge                      | 2.0751 | 0.9884 |
| Lasso                      | 2.1000 | 0.9881 |
| ElasticNet                 | 2.3079 | 0.9857 |

**Reading this through the lens of the theory above:**

1. **Plain Linear Regression already achieves R² ≈ 0.988.** That tells you the data is essentially linear and the features aren't fighting each other (no severe multicollinearity, low noise, $n \gg p$). This is the textbook case where regularization isn't needed.

2. **Ridge gives the same answer as OLS.** That makes sense — with only 5 well-conditioned features, $X^\top X$ is already invertible and stable, so adding $\alpha I$ to the diagonal barely shifts anything.

3. **Lasso loses a tiny bit.** It's shrinking the small coefficients (`Sleep Hours`, `Sample Question Papers Practiced`, `Extracurricular Activities`) toward zero. With only 5 features and all of them genuinely informative, that shrinkage costs accuracy without buying interpretability we don't already have.

4. **ElasticNet does worst.** It combines two penalties on a problem that needed neither. Classic over-engineering.

5. **Polynomial (degree 2) doesn't help.** The relationship really is linear — the squared terms and interactions don't capture additional signal.

**The lesson:** the right model is the simplest one that does the job. On clean, linear, low-dimensional data, plain linear regression *is* the answer, and the rest of these tools exist for the messy cases this dataset doesn't have.